# 06 — D3/D4 acquisition + T4 (LODO) + T3 (MAE) + T2 (4-class)
**Project:** MAG2D-NC | **Phase:** F6/F7 bridge | **Protocol:** v1.3 (frozen)

Combined experimental notebook (approved to compress the loop). Block-level
checkpointing: each task writes its results CSV on completion; reruns skip
finished blocks. Gates stop the notebook on any count/consistency failure.

Disclosed design decisions:
- **T4** cross-database work uses composition (Magpie) features only and a
  per-atom magnetization target (uB/atom) — the only representation and unit
  shared cleanly by C2DB (GPAW/PBE), JARVIS-2D (VASP/OptB88) and 2DMatPedia
  (VASP/MP). Magnetic-vs-nonmagnetic label: |M| > 0.1 uB per formula cell.
- **Overlap control:** reduced-formula matching (conservative); overlapping
  formulas are EXCLUDED from the test database in the main LODO analysis and
  included in an SI variant.
- **T3** regresses dE_zx (and dE_zy) as stored in C2DB (unit caveat printed by
  the gate cell) on the magnetic subset, Magpie features, protocol CV.
- **T2** reuses the T1 feature matrix; per frozen P7, if the DM_SS-class CI is
  vacuous the task moves to SI.

## CONFIG

In [ ]:
from pathlib import Path
from datetime import datetime
import glob

CONFIG = {
    "PROJECT_ROOT": Path.home()/"MAG2D-NC",
    "SEEDS": [0,1,2,3,4], "N_SPLITS": 5, "N_REPEATS": 3,
    "HP_TRIALS": 50, "INNER_SPLITS": 3,
    "MAG_THRESH_UB": 0.1,
}
DS = CONFIG["PROJECT_ROOT"]/"dataset"; OUT = CONFIG["PROJECT_ROOT"]/"output"
BLK = CONFIG["PROJECT_ROOT"]/"checkpoints"/"nb06_blocks"; BLK.mkdir(parents=True, exist_ok=True)
def block_done(name): return (BLK/f"{name}.done").exists()
def mark_done(name): (BLK/f"{name}.done").write_text(datetime.now().isoformat())
print("block checkpoints:", BLK)

## Dependencies

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("jarvis-tools","jarvis"), ("matminer","matminer"),
                 ("lightgbm","lightgbm"), ("requests","requests")]:
    try:
        importlib.import_module(mod); print(pkg, "OK")
    except (ImportError, OSError):
        subprocess.run([sys.executable,"-m","pip","install",pkg], check=True)
        print(pkg, "installed")

## D3 — JARVIS-2D acquisition (figshare via jarvis-tools)

In [ ]:
import pandas as pd, numpy as np

D3_PARQ = DS/"jarvis2d_raw.parquet"
if D3_PARQ.exists():
    j2 = pd.read_parquet(D3_PARQ); print("jarvis-2d cached:", len(j2))
else:
    from jarvis.db.figshare import data as jdata
    raw = jdata("dft_2d")
    rows = []
    for r in raw:
        rows.append({"jid": r.get("jid"),
                     "formula": r.get("formula"),
                     "natoms": len(r.get("atoms",{}).get("elements",[])) or np.nan,
                     "magmom": r.get("magmom_outcar", r.get("magmom_oszicar"))})
    j2 = pd.DataFrame(rows)
    j2.to_parquet(D3_PARQ, index=False)
    print("jarvis-2d downloaded:", len(j2))
j2["magmom"] = pd.to_numeric(j2["magmom"], errors="coerce")
print("magmom present:", j2.magmom.notna().sum(), "| natoms present:", j2.natoms.notna().sum())
print(j2.head(3).to_string())

## D4 — 2DMatPedia acquisition (bulk JSON)
If the automated download fails, download db.json.gz manually from
http://www.2dmatpedia.org (Download section) into dataset/ and rerun.

In [ ]:
import pandas as pd, numpy as np
from collections import Counter

D4_PARQ = DS/"twodmatpedia_raw.parquet"
if D4_PARQ.exists():
    m2 = pd.read_parquet(D4_PARQ); print("2dmatpedia cached:", len(m2))
else:
    from jarvis.db.figshare import data as jdata
    raw = jdata("twod_matpd")          # JARVIS Figshare mirror (site-independent)
    print("records downloaded:", len(raw))
    k0 = sorted(raw[0].keys())
    print("sample record keys:", k0)

    def natoms_of(r):
        st = r.get("structure") or r.get("atoms") or {}
        if isinstance(st, dict):
            if "sites" in st: return len(st["sites"])
            if "elements" in st: return len(st["elements"])
        return np.nan

    def formula_of(r):
        f = r.get("formula_pretty") or r.get("formula")
        if f: return f
        st = r.get("atoms") or r.get("structure") or {}
        els = None
        if isinstance(st, dict):
            els = st.get("elements")
            if els is None and "sites" in st:            # pymatgen-style
                els = []
                for s in st["sites"]:
                    for sp in s.get("species", []):
                        els.append(sp.get("element"))
        elif hasattr(st, "elements"):                    # jarvis Atoms object
            els = st.elements
        if els:
            c = Counter(els)
            return "".join(f"{el}{n if n>1 else ''}" for el, n in sorted(c.items()))
        return None

    rows = []
    for r in raw:
        rows.append({"mid": r.get("material_id") or r.get("jid"),
                     "formula": formula_of(r),
                     "natoms": natoms_of(r),
                     "magmom": r.get("total_magnetization")})
    m2 = pd.DataFrame(rows)
    m2.to_parquet(D4_PARQ, index=False)
    print("2dmatpedia parsed:", len(m2))

m2["magmom"] = pd.to_numeric(m2["magmom"], errors="coerce")
m2["natoms"] = pd.to_numeric(m2["natoms"], errors="coerce")
print("magmom present:", m2.magmom.notna().sum(), "| natoms present:", m2.natoms.notna().sum())
print("formula present:", m2.formula.notna().sum(), "| unique formulas:", m2.formula.nunique())
assert len(m2) > 4000, "2DMatPedia mirror smaller than expected"
assert m2.formula.notna().all(), "Formula derivation failed - inspect key list"
assert m2.formula.nunique() > 1000, "Formula diversity abnormally low - derivation faulty"
assert m2.magmom.notna().sum() > 2000, "total_magnetization empty"
print(m2.head(3).to_string())

## Unified T4 table + gates
C2DB side comes from the magnetic-wide parquet (has magnetic AND nonmagnetic
rows). All three: reduced formula, uB/atom target, binary magnetic label.

In [ ]:
import re
from math import gcd
from functools import reduce

def red_formula(f):
    d = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", str(f)):
        if el: d[el] = d.get(el, 0) + (int(n) if n else 1)
    if not d: return None
    g = reduce(gcd, d.values())
    return "".join(f"{el}{n//g if n//g>1 else ''}" for el, n in sorted(d.items()))

wide = pd.read_parquet(sorted(glob.glob(str(DS/"c2db_magnetic_wide_*.parquet")))[-1])
c2 = pd.DataFrame({"db": "c2db", "formula": wide["formula"],
                   "natoms": wide["natoms"],
                   "magmom": pd.to_numeric(wide["magmom_total"], errors="coerce")})
j2t = pd.DataFrame({"db": "jarvis2d", "formula": j2["formula"],
                    "natoms": j2["natoms"], "magmom": j2["magmom"]})
m2t = pd.DataFrame({"db": "twodmatpedia", "formula": m2["formula"],
                    "natoms": m2["natoms"], "magmom": m2["magmom"]})
T4 = pd.concat([c2, j2t, m2t], ignore_index=True)
T4["red"] = T4["formula"].map(red_formula)
T4 = T4.dropna(subset=["red","natoms","magmom"]).query("natoms>0").copy()
T4["m_abs"] = T4["magmom"].abs()
T4["m_per_atom"] = T4["m_abs"]/T4["natoms"]
T4["is_mag"] = (T4["m_abs"] > CONFIG["MAG_THRESH_UB"]).astype(int)

print(T4.groupby("db").agg(n=("red","size"), mag=("is_mag","sum"),
      m_per_atom_q95=("m_per_atom", lambda s: round(s.quantile(0.95),3))).to_string())
assert (T4.groupby("db").size() > 500).all(), "One database is suspiciously small"
print("\nGATE: all three databases loaded.")

## Overlap detection (reduced formula, conservative)

In [ ]:
sets = {db: set(g["red"]) for db, g in T4.groupby("db")}
for a in sets:
    for b in sets:
        if a < b:
            print(f"{a} ∩ {b}: {len(sets[a]&sets[b])}")
overlap_all = {}
for db in sets:
    others = set().union(*[sets[o] for o in sets if o != db])
    overlap_all[db] = sets[db] & others
    print(f"{db}: {len(overlap_all[db])}/{len(sets[db])} formulas also present in another DB")
pd.DataFrame([{"db": d, "n_overlap": len(v)} for d, v in overlap_all.items()]
             ).to_csv(OUT/"T4_overlap_stats.csv", index=False)

## Magpie features for the unified table (cached)

In [ ]:
T4F_PARQ = DS/"T4_features.parquet"
if T4F_PARQ.exists():
    T4f = pd.read_parquet(T4F_PARQ); print("T4 features cached:", T4f.shape)
else:
    from pymatgen.core import Composition
    from matminer.featurizers.composition import ElementProperty
    uniq = T4[["red"]].drop_duplicates().copy()
    uniq["composition"] = uniq["red"].map(Composition)
    ep = ElementProperty.from_preset("magpie")
    uf = ep.featurize_dataframe(uniq, col_id="composition", ignore_errors=True)
    fcols_raw = [c for c in uf.columns if c.startswith("MagpieData")]
    uf = uf.rename(columns={c: "comp_"+c.replace("MagpieData ","").replace(" ","_")
                            for c in fcols_raw})
    T4f = uf.drop(columns=["composition"])
    T4f.to_parquet(T4F_PARQ, index=False)
    print("featurized unique formulas:", T4f.shape)
FC = [c for c in T4f.columns if c.startswith("comp_")]
T4m = T4.merge(T4f, on="red", how="left").dropna(subset=FC)
print("final T4 rows:", len(T4m), "| features:", len(FC))

## T4 runs — LODO (main, overlap-excluded) + in-distribution reference
Classification: magnetic vs nonmagnetic (logreg / lgbm / dummy).
Regression (magnetic subset): m_per_atom (ridge / lgbm / mean-dummy).
Block-checkpointed; ~30-60 min total.

In [ ]:
import time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, matthews_corrcoef, mean_absolute_error, r2_score
from scipy.stats import loguniform, randint, uniform

CLS = {
 "dummy": (DummyClassifier(strategy="stratified", random_state=0), None),
 "logreg": (Pipeline([("vt",VarianceThreshold()),("sc",StandardScaler()),
                      ("clf",LogisticRegression(max_iter=3000, class_weight="balanced"))]),
            {"clf__C": loguniform(1e-3,1e3)}),
 "lgbm": (Pipeline([("vt",VarianceThreshold()),
                    ("clf",LGBMClassifier(objective="binary", verbosity=-1,
                                          class_weight="balanced", n_jobs=1))]),
          {"clf__n_estimators": randint(100,600), "clf__learning_rate": loguniform(0.01,0.3),
           "clf__num_leaves": randint(8,128), "clf__min_child_samples": randint(5,60),
           "clf__subsample": uniform(0.6,0.4), "clf__reg_lambda": loguniform(1e-3,10)}),
}
REG = {
 "dummy": (DummyRegressor(strategy="mean"), None),
 "ridge": (Pipeline([("vt",VarianceThreshold()),("sc",StandardScaler()),
                     ("reg",Ridge())]), {"reg__alpha": loguniform(1e-3,1e3)}),
 "lgbm": (Pipeline([("vt",VarianceThreshold()),
                    ("reg",LGBMRegressor(verbosity=-1, n_jobs=1))]),
          {"reg__n_estimators": randint(100,600), "reg__learning_rate": loguniform(0.01,0.3),
           "reg__num_leaves": randint(8,128), "reg__min_child_samples": randint(5,60),
           "reg__subsample": uniform(0.6,0.4), "reg__reg_lambda": loguniform(1e-3,10)}),
}

def fit_predict(models, Xtr, ytr, gtr, Xte, seed, classify):
    out = {}
    for name, (est, space) in models.items():
        t0 = time.time()
        if space is None:
            m = est; m.fit(Xtr, ytr)
        else:
            inner = (StratifiedGroupKFold if classify else GroupKFold)(CONFIG["INNER_SPLITS"])
            splits = list(inner.split(Xtr, ytr, gtr))
            s = RandomizedSearchCV(est, space, n_iter=CONFIG["HP_TRIALS"],
                                   scoring="f1_macro" if classify else "neg_mean_absolute_error",
                                   cv=splits, random_state=seed, n_jobs=-1)
            s.fit(Xtr, ytr); m = s.best_estimator_
        out[name] = m.predict(Xte)
        print(f"    {name} fitted ({time.time()-t0:.0f}s)", flush=True)
    return out

print("machinery ready (LGBM n_jobs=1, model-level progress on)")

## T4 in-distribution reference (grouped 5-fold within each DB)

In [ ]:
if not block_done("T4_indist"):
    rows = []
    for db in ("c2db","jarvis2d","twodmatpedia"):
        sub = T4m[T4m.db==db].reset_index(drop=True)
        X = sub[FC].to_numpy(); ymag = sub["is_mag"].to_numpy(); g = sub["red"].to_numpy()
        for seed in CONFIG["SEEDS"]:
            cv = StratifiedGroupKFold(CONFIG["N_SPLITS"], shuffle=True, random_state=seed)
            for tr, te in cv.split(X, ymag, g):
                preds = fit_predict(CLS, X[tr], ymag[tr], g[tr], X[te], seed, classify=True)
                for name, yp in preds.items():
                    rows.append({"task":"T4cls","scenario":f"indist_{db}","model":name,"seed":seed,
                                 "f1_macro": f1_score(ymag[te], yp, average="macro"),
                                 "mcc": matthews_corrcoef(ymag[te], yp), "n_test": len(te)})
            print(db, "seed", seed, "done", flush=True)
    pd.DataFrame(rows).to_csv(OUT/"T4_indist_results.csv", index=False)
    mark_done("T4_indist"); print("T4 in-dist block written")
else:
    print("T4_indist block already done — skipped")

## T3 — MAE (dE) regression on the C2DB magnetic subset
Gate prints the target distribution; units as stored in C2DB (verify order of
magnitude — dE values are expected in eV, small numbers).

In [ ]:
if not block_done("T3"):
    sub = wide[wide.is_magnetic == True].copy()
    for col in ("dE_zx","dE_zy"):
        sub[col] = pd.to_numeric(sub[col], errors="coerce")
    sub["red"] = sub["formula"].map(red_formula)
    sub = sub.dropna(subset=["dE_zx","red"])
    print("T3 rows:", len(sub))
    print(sub[["dE_zx","dE_zy"]].describe().round(5).to_string())
    sub = sub.merge(T4f, on="red", how="left").dropna(subset=FC)
    X = sub[FC].to_numpy(); g = sub["red"].to_numpy()
    rows = []
    for target in ("dE_zx","dE_zy"):
        yv = sub[target].to_numpy()
        for seed in CONFIG["SEEDS"]:
            for rep in range(CONFIG["N_REPEATS"]):
                cv = GroupKFold(CONFIG["N_SPLITS"])
                idx = np.random.default_rng(1000*seed+rep).permutation(len(yv))
                for tr_, te_ in cv.split(X[idx], yv[idx], g[idx]):
                    tr, te = idx[tr_], idx[te_]
                    preds = fit_predict(REG, X[tr], yv[tr], g[tr], X[te], seed, classify=False)
                    for name, yp in preds.items():
                        rows.append({"task":"T3","target":target,"model":name,"seed":seed,"rep":rep,
                                     "mae": mean_absolute_error(yv[te], yp),
                                     "r2": r2_score(yv[te], yp), "n_test": len(te)})
            print(target, "seed", seed, "done", flush=True)
    pd.DataFrame(rows).to_csv(OUT/"T3_results.csv", index=False)
    mark_done("T3"); print("T3 block written")
else:
    print("T3 block already done — skipped")

## T2 — 4-class on the 164-material set (frozen P7 rule applies)

In [ ]:
if not block_done("T2"):
    feat = pd.read_parquet(sorted(glob.glob(str(DS/"features_T1_*.parquet")))[-1])
    F1C = [c for c in feat.columns if c.startswith(("comp_","soc_","sym_"))]
    X = feat[F1C].to_numpy(); g = feat["group_id"].to_numpy()
    classes = ["FM","AFM_collinear","NC","DM_SS"]
    y4 = feat["label4"].map({c:i for i,c in enumerate(classes)}).to_numpy()

    lgbm4 = (Pipeline([("vt",VarianceThreshold()),
                       ("clf",LGBMClassifier(objective="multiclass", num_class=4,
                                             verbosity=-1, class_weight="balanced",
                                             n_jobs=1))]),
             CLS["lgbm"][1])          # same HP space - equal budget preserved

    rows, recalls = [], {c: [] for c in classes}
    from sklearn.metrics import recall_score
    for seed in CONFIG["SEEDS"]:
        for rep in range(CONFIG["N_REPEATS"]):
            cv = StratifiedGroupKFold(CONFIG["N_SPLITS"], shuffle=True,
                                      random_state=1000*seed+rep)
            for tr, te in cv.split(X, y4, g):
                preds = fit_predict({"logreg": CLS["logreg"], "lgbm": lgbm4,
                                     "dummy": CLS["dummy"]},
                                    X[tr], y4[tr], g[tr], X[te], seed, classify=True)
                for name, yp in preds.items():
                    rows.append({"task":"T2","model":name,"seed":seed,"rep":rep,
                                 "f1_macro": f1_score(y4[te], yp, average="macro"),
                                 "mcc": matthews_corrcoef(y4[te], yp)})
                    if name == "lgbm":
                        rc = recall_score(y4[te], yp, average=None,
                                          labels=list(range(4)), zero_division=0)
                        for c, r_ in zip(classes, rc):
                            if (y4[te]==classes.index(c)).sum() > 0:
                                recalls[c].append(r_)
        print("T2 seed", seed, "done", flush=True)

    pd.DataFrame(rows).to_csv(OUT/"T2_results.csv", index=False)
    import json as _j2
    (OUT/"T2_perclass_recall.json").write_text(_j2.dumps(
        {c: {"mean": float(np.mean(v)), "n": len(v), "values": [float(x) for x in v]}
         for c, v in recalls.items()}, indent=1))
    rng = np.random.default_rng(0)
    print("\nT2 per-class recall (lgbm, fold-level bootstrap 95% CI):")
    p7 = {}
    for c in classes:
        v = np.array(recalls[c])
        bs = rng.choice(v, size=(5000, len(v)), replace=True).mean(axis=1)
        lo, hi = np.percentile(bs, [2.5, 97.5])
        p7[c] = (round(float(lo),3), round(float(hi),3))
        print(f"  {c:14s} mean={v.mean():.3f} CI=[{lo:.3f},{hi:.3f}] (n_folds={len(v)})")
    dm_lo, dm_hi = p7["DM_SS"]
    verdict = "MAIN TEXT" if (dm_hi - dm_lo) < 0.9 else "MOVE TO SI (frozen P7 rule)"
    print("\nP7 DM_SS verdict:", verdict)
    mark_done("T2"); print("T2 block written")
else:
    print("T2 block already done -- skipped")

## Wrap-up: summaries to paste back

In [ ]:
import numpy as np
print("=== T4 LODO (main) ===")
t4 = pd.read_csv(OUT/"T4_lodo_results.csv")
print(t4[t4.task=="T4cls"].groupby(["scenario","model"])["f1_macro"].agg(["mean","std"]).round(3).to_string())
print(t4[t4.task=="T4reg"].groupby(["scenario","model"])[["mae_uB","r2"]].mean().round(3).to_string())
print("\n=== T4 in-dist ===")
ti = pd.read_csv(OUT/"T4_indist_results.csv")
print(ti.groupby(["scenario","model"])["f1_macro"].agg(["mean","std"]).round(3).to_string())
print("\n=== T3 ===")
t3 = pd.read_csv(OUT/"T3_results.csv")
print(t3.groupby(["target","model"])[["mae","r2"]].mean().round(4).to_string())
print("\n=== T2 ===")
t2 = pd.read_csv(OUT/"T2_results.csv")
print(t2.groupby("model")[["f1_macro","mcc"]].agg(["mean","std"]).round(3).to_string())
print("\nRun the manifest+backup cell (nb04) after this.")